In [0]:
%pip install langchain langchain-tavily databricks-langchain tavily-python ddgs -q

In [0]:
dbutils.library.restartPython()

In [0]:
from typing import List
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_core.messages import HumanMessage
from databricks_langchain import ChatDatabricks
from langchain_tavily import TavilySearch
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.prompts import PromptTemplate
from IPython.display import Markdown

In [0]:
import langchain
langchain.__version__

In [0]:
import os
os.environ['TAVILY_API_KEY'] = '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'

In [0]:
import mlflow
# MLflow autologging bekapcsolása LangChain-hez
mlflow.langchain.autolog()

In [0]:
class Source(BaseModel):
    """Schema for a source used by the agent"""

    url: str = Field(description="The URL of the source")

In [0]:
class AgentResponse(BaseModel):
    """Schema for agent response with answer and sources"""

    answer: str = Field(description="Thr agent's answer to the query")
    sources: List[Source] = Field(
        default_factory=list, description="List of sources used to generate the answer"
    )

In [0]:
model_4 = 'databricks-meta-llama-3-1-8b-instruct'
model_3 = 'databricks-llama-4-maverick'
model_gpt_oss = 'databricks-gpt-oss-120b'  # ez a legjobb

In [0]:
llm = ChatDatabricks(endpoint=model_gpt_oss)
# Ingyenes keresők (nem kell API kulcs):
tool_duck   = [DuckDuckGoSearchRun()]  # Általános webes keresés
tool_tavily = [TavilySearch()]  # API kulcs kell (TAVILY_API_KEY)

# Mindkét agent létrehozása
agent_duck   = create_agent(llm, tool_duck)
agent_tavily = create_agent(llm, tool_tavily)

In [0]:
# Kérdés: OpenAI legújabb hírei
query = 'What are the latest news about OpenAI and their newest models?'
result = agent_tavily.invoke({"messages": HumanMessage(content=query)})
# Markdown formátumban renderelve (implicit display)
Markdown(result['messages'][-1].content)

In [0]:
# Kérdés: AI Engineer állások San Franciscóban
query = 'Search for 3 job postings in San Francisco for an AI engineer with langchain knowledge. Please present the job details too.'
result_jobs = agent_duck.invoke({"messages": HumanMessage(content=query)})

Markdown(result_jobs['messages'][-1].content)

In [0]:
# Egyéb ingyenes keresők példái:

# Wikipedia keresés
# from langchain_community.tools import WikipediaQueryRun
# from langchain_community.utilities import WikipediaAPIWrapper
# wikipedia = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())

# Tudományos cikkek (Arxiv)
# from langchain_community.tools import ArxivQueryRun
# arxiv = ArxivQueryRun()

# Orvosi cikkek (Pubmed)
# from langchain_community.tools import PubmedQueryRun
# pubmed = PubmedQueryRun()

# Több eszköz együtt:
# tools = [DuckDuckGoSearchRun(), wikipedia, arxiv]

In [0]:
# Kérdés: EUR/HUF árfolyam
query = 'What is the current exchange rate between EUR and HUF?'
result_eur = agent_tavily.invoke({"messages": HumanMessage(content=query)})

Markdown(result_eur['messages'][-1].content)

In [0]:
# Kérdés: Python vizualizációs könyvtárak
query = 'Find the top 3 Python libraries for data visualization in 2026 with examples'
result_viz = agent_duck.invoke({"messages": HumanMessage(content=query)})

Markdown(result_viz['messages'][-1].content)

In [0]:
# Kérdés: LangChain vs LlamaIndex
query = 'What are the main differences between LangChain and LlamaIndex frameworks?'
result_compare = agent_tavily.invoke({"messages": HumanMessage(content=query)})

Markdown(result_compare['messages'][-1].content)

In [0]:
# Kérdés: Londoni időjárás
query = 'What is the current weather in London? Today is 2026.04.11! In short, please.'
#Please provide detailed information.
result_london = agent_duck.invoke({"messages": HumanMessage(content=query)})

Markdown(result_london['messages'][-1].content)

In [0]:
# Kérdés: Londoni időjárás
query = 'What is the current weather in London? Today is 2026.04.11! In short, please.'
#Please provide detailed information.
result_london = agent_tavily.invoke({"messages": HumanMessage(content=query)})

Markdown(result_london['messages'][-1].content)